# A2: validate every cited number
Checks each claim in `data/claims.json`: the value at its `numbers.json` path (written by A1 from the saved results), that the cited text appears in `ARTICLE.md`, and for report-sourced claims that the pattern appears in the source report. Writes `CLAIMS.md`.

In [ ]:
import glob, json, os, re
from pathlib import Path
ART = Path("/Users/isaac/App_Files/Code/ez-penta/post_train_3/article")
REPO = ART.parents[1]
PT3D = Path(glob.glob(os.path.expanduser("~/Library/CloudStorage/GoogleDrive-*"))[0]) / "My Drive" / "ez-penta" / "post_train_3"
N = json.loads((ART / "data" / "numbers.json").read_text())
CL = json.loads((ART / "data" / "claims.json").read_text())
TXT = (ART / "ARTICLE.md").read_text()
norm = lambda s: re.sub(r"\s+", " ", s)
TXTn = norm(TXT)
def src_text(source):
    for base in (REPO, PT3D):
        p = base / source.replace("post_train_3/", "", 1) if base == PT3D else base / source
        if p.exists():
            return p.read_text()
    return None
rows, bad = [], 0
for c in CL:
    in_article = norm(c["text"]) in TXTn
    if c["kind"] == "num":
        v = N
        for k in c["path"]:
            v = v[k]
        ok_val = round(float(v), c["dp"]) == round(float(c["value"]), c["dp"])
        status = "ok" if (ok_val and in_article) else f"FAIL (value {v}, in article {in_article})"
        src = f"`numbers.json` → {' → '.join(c['path'])} (from {c['source']})"
    else:
        if c["pattern"] is None:
            ok_src, status_note = True, "counted in session, no saved file"
        else:
            t = src_text(c["source"]); ok_src = t is not None and norm(c["pattern"]) in norm(t)
        status = "ok" if (ok_src and in_article) else f"FAIL (source match {ok_src}, in article {in_article})"
        src = f"{c['source']}" + (f': "{c["pattern"]}"' if c["pattern"] else " (manual count)")
    bad += status != "ok"
    rows.append(f"| {c['claim']} | {c['text'].replace(chr(10), ' ').replace('|', '¦')} | {src.replace('|', '¦')} | {status} |")
md = ["# Claim audit: v2 article", "",
      f"Generated by `notebooks/A2_validate.ipynb`. {len(CL)} claims checked, {bad} failed. Numerical claims are compared with "
      "`data/numbers.json`, which `notebooks/A1_figures.ipynb` builds from the saved results; text claims are matched in the source report.", "",
      "| Claim | Text in article | Source | Check |", "|---|---|---|---|"] + rows
(ART / "CLAIMS.md").write_text("\n".join(md) + "\n")
print(f"{len(CL)} claims, {bad} failed")
for r in rows:
    if "FAIL" in r: print(r)
assert bad == 0
